# 🔬 Multimodal RAG with ColPali & Late-Interaction Retrieval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/multi_model_rag_with_colpali.ipynb)

> **Beyond captioning: embedding document *images* directly into a shared vector space with queries.**

| Component | Choice |
|---|---|
| LLM | `Qwen/Qwen3-8B` (4-bit NF4) |
| Text Embeddings | `BAAI/bge-base-en-v1.5` (sentence-transformers) |
| Vision Embeddings | `vidore/colpali-v1.3` → fallback: `openai/clip-vit-base-patch32` |
| Vector Store | FAISS (flat & IVF) |
| Synthetic Data | Pillow-generated document page images |

This notebook is a **deep educational walkthrough** of late-interaction retrieval — the mechanism that
makes ColPali possible. We build every component from scratch, compare it against single-vector
baselines, and wire up a full multimodal RAG pipeline.

<img src="../images/multi_model_rag_with_colpali.svg" alt="ColPali-RAG" width="400">

## 1 · Environment Setup

We install only open-source, pip-installable packages — **no LangChain, no LlamaIndex, no OpenAI API**.
Everything runs locally on the GPU.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy Pillow

### Imports & Model Cache

We point the Hugging Face cache to Google Drive so models survive Colab restarts.
This avoids re-downloading the ~8 GB Qwen model every session.

In [ ]:
import os, sys, time, textwrap, warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR = '/content/drive/MyDrive/models'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME']            = CACHE_DIR
os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR

import numpy as np
import torch
import faiss
from PIL import Image, ImageDraw, ImageFont

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  •  Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}  •  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

---
## 2 · Beyond Captioning: Why Embed Images Directly?

The **captioning-based** multimodal RAG pipeline (covered in our companion notebook) works like this:

```
Document Page  →  OCR / VLM Caption  →  Text Chunk  →  Text Embedding  →  FAISS
```

This is elegant — but it **loses information at every arrow**:

| Stage | What's Lost |
|---|---|
| OCR | Layout, font emphasis, spatial relationships |
| Captioning | Subjective — model decides what's "important" |
| Chunking | Table structure, figure context, cross-references |
| Single embedding | Fine-grained token-level matches |

### The Alternative: Visual Embeddings

What if we could take a **screenshot of the page** and embed it *directly* into the same vector
space as the user's text query?  No OCR, no captioning, no chunking.

```
Document Page Image  →  Vision Encoder  →  Token-level Embeddings  →  Late Interaction Score
Query Text           →  Text Encoder    →  Token-level Embeddings  ↗
```

This is exactly what **ColPali** does. Let's understand the theory before we code it.

---
## 3 · What is ColPali?

**ColPali** (2024, Faysse et al.) is a **vision-language retrieval model** that produces
*token-level* embeddings from document page images. The name reveals its lineage:

- **Col** → from **ColBERT** (Khattab & Zaharia, 2020), the late-interaction text retrieval model
- **Pali** → from **PaLI** (Chen et al., 2022), Google's vision-language model family

### Key Ideas

1. **Page-as-image**: The entire document page (text, tables, figures, layout) is treated as a
   single image. No OCR is needed.

2. **Token-level embeddings**: Instead of producing one vector per page, ColPali produces a
   *matrix* of embeddings — one vector per image patch (analogous to one vector per text token
   in ColBERT).

3. **Late interaction scoring**: Query and document embeddings interact *after* encoding, via
   the MaxSim operator. This captures fine-grained matches that a single cosine similarity misses.

4. **Shared embedding space**: Both the text query and the image page live in the same
   $d$-dimensional space, so we can compute meaningful similarities between them.

### Architecture at a Glance

```
          ┌──────────────┐
  Query → │ Text Encoder │ → Q ∈ ℝ^(m × d)     (m query tokens)
          └──────────────┘
                                                  MaxSim(Q, D) → score
          ┌───────────────┐
  Page  → │ Vision Encoder│ → D ∈ ℝ^(n × d)     (n image patches)
          └───────────────┘
```

---
## 4 · Late Interaction Explained

### The Problem with Single-Vector Retrieval

In standard dense retrieval (DPR, BGE, etc.), each document is compressed into **one vector**.
The query is also one vector.  Retrieval = cosine similarity.

This works well for short passages, but for *pages* containing multiple topics, tables, and
figures, a single vector is a lossy bottleneck. A query about "Table 3's F1 scores" and a
query about "the abstract's main claim" would need to match the *same* page vector.

### Late Interaction: Token-Level Matching

**Late interaction** keeps *all* token embeddings and computes a richer score:

$$\text{MaxSim}(Q, D) = \sum_{i=1}^{m} \max_{j=1}^{n} \; Q_i \cdot D_j$$

In words:
1. For **each query token** $Q_i$, find the **most similar document token** $D_j$ (the max dot
   product across all $n$ document tokens).
2. **Sum** these maximum similarities across all $m$ query tokens.

### Why This Works

- Each query token independently finds its best match in the document.
- A query about "F1 scores in Table 3" will match specific *patches* showing the table, even if
  the rest of the page is about something else entirely.
- It's **more expressive** than cosine similarity but **cheaper** than full cross-attention
  (which would require feeding Q and D through a transformer together).

### Interaction Taxonomy

| Approach | Interaction | Vectors per doc | Expressiveness | Cost |
|---|---|---|---|---|
| Bi-encoder (DPR/BGE) | None (cosine) | 1 | Low | Cheapest |
| **Late interaction (ColBERT/ColPali)** | **MaxSim** | **n (token-level)** | **Medium** | **Medium** |
| Cross-encoder (BERT reranker) | Full attention | N/A | High | Expensive |

### Let's Implement MaxSim from Scratch

Before touching any model, let's implement the scoring function in pure NumPy
so we can see exactly how late interaction works.

In [ ]:
def maxsim_score(Q: np.ndarray, D: np.ndarray) -> float:
    """
    Late-interaction MaxSim score.
    Q : (m, d) — m query  token embeddings of dimension d
    D : (n, d) — n document token embeddings of dimension d
    Returns: scalar score = Σ_i max_j (Q_i · D_j)
    """
    # (m, d) @ (d, n) → (m, n)  similarity matrix
    sim_matrix = Q @ D.T
    # For each query token, find the max similarity to any document token
    max_sims = sim_matrix.max(axis=1)    # shape (m,)
    return float(max_sims.sum())


def cosine_score(q_vec: np.ndarray, d_vec: np.ndarray) -> float:
    """Standard single-vector cosine similarity."""
    return float(np.dot(q_vec, d_vec) / (np.linalg.norm(q_vec) * np.linalg.norm(d_vec) + 1e-10))


# ── Quick sanity check with random embeddings ──
np.random.seed(42)
d = 128
Q_rand = np.random.randn(5, d).astype(np.float32)   # 5 query  tokens
D_rand = np.random.randn(20, d).astype(np.float32)   # 20 doc tokens

ms = maxsim_score(Q_rand, D_rand)
cs = cosine_score(Q_rand.mean(axis=0), D_rand.mean(axis=0))

print(f'MaxSim score (token-level):     {ms:.4f}')
print(f'Cosine score (mean-pool):       {cs:.4f}')
print(f'\nMaxSim uses {Q_rand.shape[0]}×{D_rand.shape[0]} = {Q_rand.shape[0]*D_rand.shape[0]} comparisons')
print(f'Cosine uses 1×1 = 1 comparison')
print('\n→ MaxSim captures fine-grained token matches that cosine misses.')

In [ ]:
np.random.seed(7)
d = 64

# ── Demonstrating MaxSim's Advantage ──
# Craft a scenario: document with TWO distinct topics.
# MaxSim matches the query to the relevant portion; cosine gets confused.

# Create two distinct 'topic' directions
topic_a = np.random.randn(d).astype(np.float32)
topic_a /= np.linalg.norm(topic_a)
topic_b = np.random.randn(d).astype(np.float32)
topic_b /= np.linalg.norm(topic_b)

# Document: 10 tokens about topic A, 10 tokens about topic B (+ noise)
noise = 0.3
D_mixed = np.vstack([
    topic_a + noise * np.random.randn(10, d).astype(np.float32),   # tokens 0-9:  topic A
    topic_b + noise * np.random.randn(10, d).astype(np.float32),   # tokens 10-19: topic B
])

# Query about topic A only (3 tokens)
Q_a = topic_a + noise * np.random.randn(3, d).astype(np.float32)

# Query about topic B only (3 tokens)
Q_b = topic_b + noise * np.random.randn(3, d).astype(np.float32)

# ── MaxSim scores ──
ms_a = maxsim_score(Q_a, D_mixed)
ms_b = maxsim_score(Q_b, D_mixed)

# ── Cosine scores (mean-pooled) ──
d_mean = D_mixed.mean(axis=0)
cs_a = cosine_score(Q_a.mean(axis=0), d_mean)
cs_b = cosine_score(Q_b.mean(axis=0), d_mean)

print('Document has tokens about Topic A (first half) and Topic B (second half).')
print()
print(f'Query about Topic A:  MaxSim = {ms_a:7.3f}   Cosine = {cs_a:.4f}')
print(f'Query about Topic B:  MaxSim = {ms_b:7.3f}   Cosine = {cs_b:.4f}')
print()
print('MaxSim difference:  ', f'{abs(ms_a - ms_b):.3f}  (can distinguish topics)')
print('Cosine difference:  ', f'{abs(cs_a - cs_b):.4f}  (topics blurred together)')
print()
print('→ MaxSim finds the RELEVANT tokens; cosine averages everything into mush.')

---
## 5 · ColBERT Architecture — The Text-Only Predecessor

ColBERT (Khattab & Zaharia, SIGIR 2020) introduced late interaction for **text retrieval**:

```
  Query:    "What is attention?"      →  BERT encoder  →  Q ∈ ℝ^(m × 128)
  Document: "Attention is a mechanism  →  BERT encoder  →  D ∈ ℝ^(n × 128)
             that allows models to..."
```

### Key Design Decisions

1. **Separate encoders** — query and document are encoded *independently*. This means documents
   can be pre-computed offline, and only the query needs encoding at search time.

2. **Low-dimensional projections** — Token embeddings are projected from BERT's 768-d to 128-d.
   This makes the (m × n) similarity matrix cheap to compute.

3. **Token-level storage** — Unlike DPR which stores 1 vector per passage, ColBERT stores
   ~100-200 vectors per passage. More storage, but much richer matching.

4. **MaxSim scoring** — exactly the function we implemented above.

### Efficiency Tricks

- **Pre-computation**: All document embeddings are computed offline and stored.
- **Candidate generation**: A fast first stage (e.g., BM25 or single-vector ANN) retrieves
  candidates, then MaxSim re-ranks them.
- **Compression**: ColBERTv2 uses residual compression to shrink token embeddings from 128-d
  floats to ~16 bytes per token.

---
## 6 · ColPali: From Text Tokens to Image Patches

ColPali's key insight: **replace the text document encoder with a vision encoder**, keeping
everything else (query encoder, late interaction, MaxSim) the same.

```
  ColBERT:                           ColPali:
  ┌────────────┐                     ┌────────────────────┐
  │ BERT (text) │ → doc tokens       │ PaLI (vision)       │ → image patch embeddings
  └────────────┘                     └────────────────────┘
  ┌────────────┐                     ┌────────────────────┐
  │ BERT (text) │ → query tokens     │ PaLI (text encoder) │ → query tokens
  └────────────┘                     └────────────────────┘
         │                                    │
         ▼                                    ▼
     MaxSim(Q, D)                         MaxSim(Q, D)
```

### What Changes?

| Aspect | ColBERT | ColPali |
|---|---|---|
| Document input | Raw text | Page screenshot (image) |
| Document encoder | BERT | PaLI-based Vision Transformer |
| Tokens = | Text subwords | Image patches (e.g., 16×16 px) |
| Query input | Raw text | Raw text (same) |
| Query encoder | BERT | PaLI text encoder |
| Scoring | MaxSim | MaxSim (identical) |
| OCR needed? | N/A (text input) | **No** — layout preserved visually |

### Why This Matters

- **Tables**: A table in an image is encoded as a grid of patches. A query about "row 3, column 5"
  can match the specific patches at that location.
- **Figures**: Charts and diagrams are naturally encoded — no need to describe them in text.
- **Layout**: Headers, footnotes, sidebars — all preserved in the visual encoding.
- **Multilingual**: Works on any script without language-specific OCR.

---
## 7 · Creating Synthetic Document Page Images

Since we don't want to depend on external PDFs, we'll generate realistic-looking
document page images using PIL. Each page simulates a different type of content:
tables, text paragraphs, charts (as pixel art), and mixed layouts.

In [ ]:
def create_text_page(title: str, paragraphs: list[str], page_num: int = 1) -> Image.Image:
    """Create a document page image with title and text paragraphs."""
    W, H = 800, 1100
    img = Image.new('RGB', (W, H), 'white')
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 28)
        font_body  = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 16)
        font_small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 12)
    except OSError:
        font_title = ImageFont.load_default()
        font_body  = font_title
        font_small = font_title

    # Header line
    draw.line([(50, 60), (W-50, 60)], fill='navy', width=2)
    draw.text((50, 20), title, fill='navy', font=font_title)
    draw.line([(50, 70), (W-50, 70)], fill='navy', width=1)

    y = 90
    for para in paragraphs:
        wrapped = textwrap.wrap(para, width=80)
        for line in wrapped:
            if y > H - 80:
                break
            draw.text((60, y), line, fill='black', font=font_body)
            y += 22
        y += 12

    # Footer
    draw.line([(50, H-50), (W-50, H-50)], fill='gray', width=1)
    draw.text((W//2 - 10, H-45), str(page_num), fill='gray', font=font_small)

    return img


def create_table_page(title: str, headers: list[str], rows: list[list[str]], page_num: int = 1) -> Image.Image:
    """Create a document page with a data table."""
    W, H = 800, 1100
    img = Image.new('RGB', (W, H), 'white')
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
        font_hdr   = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 14)
        font_cell  = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 13)
        font_small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 12)
    except OSError:
        font_title = ImageFont.load_default()
        font_hdr = font_cell = font_small = font_title

    draw.text((50, 30), title, fill='navy', font=font_title)
    draw.line([(50, 65), (W-50, 65)], fill='navy', width=2)

    ncols = len(headers)
    col_w = (W - 120) // ncols
    x0, y0 = 60, 90

    # Header row
    draw.rectangle([x0, y0, x0 + ncols*col_w, y0+30], fill='#dde4f0')
    for ci, h in enumerate(headers):
        draw.text((x0 + ci*col_w + 8, y0+7), h, fill='navy', font=font_hdr)
    y = y0 + 30

    for ri, row in enumerate(rows):
        bg = '#f5f5f5' if ri % 2 == 0 else 'white'
        draw.rectangle([x0, y, x0 + ncols*col_w, y+26], fill=bg)
        for ci, val in enumerate(row):
            draw.text((x0 + ci*col_w + 8, y+5), str(val), fill='black', font=font_cell)
        draw.line([(x0, y+26), (x0+ncols*col_w, y+26)], fill='#cccccc', width=1)
        y += 26

    # Grid lines
    for ci in range(ncols+1):
        draw.line([(x0+ci*col_w, y0), (x0+ci*col_w, y)], fill='#999999', width=1)

    draw.line([(50, H-50), (W-50, H-50)], fill='gray', width=1)
    draw.text((W//2-10, H-45), str(page_num), fill='gray', font=font_small)
    return img


def create_chart_page(title: str, data: dict, page_num: int = 1) -> Image.Image:
    """Create a page with a simple bar chart drawn in PIL."""
    W, H = 800, 1100
    img = Image.new('RGB', (W, H), 'white')
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
        font_label = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 14)
        font_small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 12)
    except OSError:
        font_title = ImageFont.load_default()
        font_label = font_small = font_title

    draw.text((50, 30), title, fill='navy', font=font_title)
    draw.line([(50, 65), (W-50, 65)], fill='navy', width=2)

    labels = list(data.keys())
    values = list(data.values())
    max_val = max(values)
    n = len(labels)
    chart_x, chart_y, chart_w, chart_h = 100, 120, 580, 400
    bar_w = chart_w // (n * 2)
    colors = ['#3366cc', '#dc3912', '#ff9900', '#109618', '#990099', '#0099c6']

    # Y axis
    draw.line([(chart_x, chart_y), (chart_x, chart_y+chart_h)], fill='black', width=2)
    # X axis
    draw.line([(chart_x, chart_y+chart_h), (chart_x+chart_w, chart_y+chart_h)], fill='black', width=2)

    for i, (label, val) in enumerate(zip(labels, values)):
        bh = int((val / max_val) * (chart_h - 20))
        bx = chart_x + 30 + i * (chart_w // n)
        by = chart_y + chart_h - bh
        draw.rectangle([bx, by, bx+bar_w, chart_y+chart_h], fill=colors[i % len(colors)])
        draw.text((bx, chart_y+chart_h+5), label, fill='black', font=font_label)
        draw.text((bx+2, by-18), f'{val:.1f}', fill='black', font=font_label)

    draw.line([(50, H-50), (W-50, H-50)], fill='gray', width=1)
    draw.text((W//2-10, H-45), str(page_num), fill='gray', font=font_small)
    return img


print('✓ Page generation functions defined.')

In [ ]:
# ── Synthetic document: a research paper on "Neural Architecture Search" ──

page1 = create_text_page('Neural Architecture Search: A Survey', [
    'Abstract: Neural Architecture Search (NAS) automates the design of neural network architectures. '
    'This survey covers the three main components of NAS: the search space, the search strategy, '
    'and the performance estimation strategy. We review over 200 papers published between 2016 and 2024.',
    'The search space defines which architectures can be represented. Common choices include chain-structured '
    'networks, multi-branch networks, and cell-based search spaces where a small cell is repeated.',
    'Search strategies range from reinforcement learning (Zoph & Le, 2017) to evolutionary algorithms '
    '(Real et al., 2019) and gradient-based methods like DARTS (Liu et al., 2019).',
    'Performance estimation is the bottleneck: training each candidate to convergence is expensive. '
    'Weight sharing, early stopping, and predictor-based methods address this.',
], page_num=1)

page2 = create_table_page('Table 1: NAS Method Comparison',
    headers=['Method', 'Search Cost (GPU-days)', 'CIFAR-10 Acc (%)', 'ImageNet Top-1 (%)'],
    rows=[
        ['NASNet',     '1800',  '97.35', '74.0'],
        ['ENAS',       '0.5',   '97.11', '—'   ],
        ['DARTS',      '1.5',   '97.24', '73.3'],
        ['ProxylessNAS','8.3',  '—',     '75.1'],
        ['EfficientNet','—',    '—',     '84.3'],
        ['OFA',        '2.0',   '—',     '80.0'],
        ['FBNet',      '9.0',   '—',     '74.9'],
        ['MnasNet',    '288',   '—',     '75.2'],
    ], page_num=2)

page3 = create_chart_page('Figure 1: Search Cost vs Accuracy',
    data={'NASNet': 74.0, 'ENAS': 73.0, 'DARTS': 73.3, 'ProxylessNAS': 75.1,
          'EfficientNet': 84.3, 'OFA': 80.0}, page_num=3)

page4 = create_text_page('Weight Sharing and One-Shot Methods', [
    'One-shot NAS methods train a single supernet that contains all candidate architectures as subgraphs. '
    'At search time, different subgraphs are sampled and evaluated without retraining from scratch.',
    'The key advantage is dramatic reduction in search cost: from thousands of GPU-days to single digits. '
    'However, weight sharing introduces coupling between architectures — the supernet weights may not '
    'accurately reflect standalone performance.',
    'Recent work on few-shot NAS (Zhao et al., 2021) splits the supernet into several sub-supernets '
    'to reduce interference while keeping the cost manageable.',
    'Hardware-aware NAS extends these methods by incorporating latency, memory, and energy constraints '
    'directly into the search objective, enabling deployment on edge devices.',
], page_num=4)

page5 = create_text_page('Conclusions and Future Directions', [
    'NAS has matured from a compute-intensive curiosity into a practical tool. Key future directions include:',
    '1. Transferable architectures: Designing search spaces that generalize across tasks and datasets.',
    '2. Beyond classification: Extending NAS to detection, segmentation, NLP, and multimodal tasks.',
    '3. Green AI: Reducing the carbon footprint of architecture search through efficient methods.',
    '4. Foundation model adaptation: Using NAS to find optimal fine-tuning configurations for LLMs.',
    '5. Theoretical understanding: Connecting NAS to neural scaling laws and loss landscape geometry.',
], page_num=5)

doc_pages = [page1, page2, page3, page4, page5]
page_descriptions = [
    'Abstract and introduction to Neural Architecture Search survey',
    'Table comparing NAS methods: cost, CIFAR-10, ImageNet accuracy',
    'Bar chart of search cost vs ImageNet accuracy for NAS methods',
    'Weight sharing, one-shot NAS, hardware-aware search',
    'Conclusions: transferable architectures, green AI, future directions',
]

print(f'Created {len(doc_pages)} synthetic document pages.')
for i, desc in enumerate(page_descriptions):
    print(f'  Page {i+1}: {desc}')

In [ ]:
from IPython.display import display

for i, page in enumerate(doc_pages):
    print(f'\n── Page {i+1}: {page_descriptions[i]} ──')
    display(page.resize((400, 550)))

---
## 8 · Text Embeddings with BGE

Before tackling vision embeddings, let's set up our text embedding baseline with
`BAAI/bge-base-en-v1.5`. We'll use this both as our single-vector baseline and to
demonstrate late interaction with text tokens.

In [ ]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer(
    'BAAI/bge-base-en-v1.5',
    cache_folder=CACHE_DIR,
    device=device
)
print(f'Loaded BGE model  •  Embedding dim: {text_model.get_sentence_embedding_dimension()}')
print(f'Device: {text_model.device}')

In [ ]:
# ── Single-Vector Baseline Retrieval ──
# Standard approach: embed each page description → single vector → FAISS cosine sim.

# Embed page descriptions as single vectors
page_embeddings = text_model.encode(page_descriptions, normalize_embeddings=True)
print(f'Page embeddings shape: {page_embeddings.shape}')  # (5, 768)

# Build FAISS index
dim = page_embeddings.shape[1]
index_flat = faiss.IndexFlatIP(dim)  # Inner product = cosine sim (since normalized)
index_flat.add(page_embeddings.astype(np.float32))
print(f'FAISS index size: {index_flat.ntotal} vectors of dim {dim}')

In [ ]:
queries = [
    'What is the ImageNet accuracy of DARTS?',
    'How much does NASNet cost in GPU days?',
    'What are the future directions for NAS?',
    'How does weight sharing reduce search cost?',
    'Show me a chart comparing NAS methods',
]

print('=== Single-Vector Retrieval (Cosine Similarity) ===')
print()

for q in queries:
    q_emb = text_model.encode([q], normalize_embeddings=True)
    scores, ids = index_flat.search(q_emb.astype(np.float32), k=3)
    print(f'Q: "{q}"')
    for rank, (score, idx) in enumerate(zip(scores[0], ids[0])):
        print(f'   #{rank+1}  (score={score:.4f}) Page {idx+1}: {page_descriptions[idx]}')
    print()

---
## 9 · Late Interaction with Text Token Embeddings

Now let's implement **ColBERT-style late interaction** using our BGE model's internal
token representations (before mean-pooling). This demonstrates exactly how ColBERT/ColPali
scoring works — token by token.

We extract the hidden states from the transformer and use them as per-token embeddings,
then apply our MaxSim scoring function.

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Load the underlying transformer (same model as sentence-transformers uses)
bge_tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5', cache_dir=CACHE_DIR)
bge_model = AutoModel.from_pretrained('BAAI/bge-base-en-v1.5', cache_dir=CACHE_DIR).to(device)
bge_model.eval()

def get_token_embeddings(text: str) -> np.ndarray:
    """Extract per-token embeddings from BGE (before pooling)."""
    inputs = bge_tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = bge_model(**inputs)
    # last_hidden_state: (1, seq_len, 768)
    token_embs = outputs.last_hidden_state[0].cpu().numpy()
    # L2-normalize each token
    norms = np.linalg.norm(token_embs, axis=1, keepdims=True) + 1e-10
    return (token_embs / norms).astype(np.float32)

# Quick test
test_embs = get_token_embeddings('What is attention?')
print(f'Token embeddings for "What is attention?": shape {test_embs.shape}')
print(f'  → {test_embs.shape[0]} tokens, {test_embs.shape[1]} dimensions each')

In [ ]:
# Pre-compute token-level embeddings for each page description
doc_token_embs = []
for i, desc in enumerate(page_descriptions):
    embs = get_token_embeddings(desc)
    doc_token_embs.append(embs)
    print(f'Page {i+1}: {embs.shape[0]:3d} tokens  →  embedding matrix {embs.shape}')

print(f'\nTotal token vectors stored: {sum(e.shape[0] for e in doc_token_embs)}')
print('(vs 5 vectors in single-vector mode — late interaction uses ~20x more storage)')

In [ ]:
print('=== Late Interaction Retrieval (MaxSim) ===')
print()

for q in queries:
    Q = get_token_embeddings(q)  # (m, 768)
    scores = []
    for idx, D in enumerate(doc_token_embs):
        s = maxsim_score(Q, D)
        scores.append((s, idx))
    scores.sort(reverse=True)

    print(f'Q: "{q}"')
    for rank, (score, idx) in enumerate(scores[:3]):
        print(f'   #{rank+1}  (MaxSim={score:8.3f}) Page {idx+1}: {page_descriptions[idx]}')
    print()

### Head-to-Head: Single-Vector vs Late Interaction

Let's compare the two approaches side by side. We'll look at whether the
top-1 result matches for each query and highlight any differences.

In [ ]:
print(f'{"Query":<50} {"Cosine Top-1":>15} {"MaxSim Top-1":>15} {"Match?":>8}')
print('─' * 95)

for q in queries:
    # Single-vector
    q_emb = text_model.encode([q], normalize_embeddings=True)
    _, cos_ids = index_flat.search(q_emb.astype(np.float32), k=1)
    cos_top = cos_ids[0][0] + 1

    # Late interaction
    Q = get_token_embeddings(q)
    ms_scores = [(maxsim_score(Q, D), idx) for idx, D in enumerate(doc_token_embs)]
    ms_top = sorted(ms_scores, reverse=True)[0][1] + 1

    match = '✓' if cos_top == ms_top else '✗ DIFF'
    print(f'{q:<50} {"Page "+str(cos_top):>15} {"Page "+str(ms_top):>15} {match:>8}')

print()
print('Note: With short text descriptions, both methods often agree.')
print('The advantage of late interaction grows with longer, multi-topic documents.')

---
## 10 · Vision Embeddings: Bridging Images and Text

Now we get to the core of ColPali's approach: embedding **images** into the same
space as text queries.

We'll try to load `vidore/colpali-v1.3` first. If it's too large for our GPU,
we'll demonstrate the concept using CLIP (`openai/clip-vit-base-patch32`),
which is a smaller vision-language model that also maps images and text to a
shared embedding space.

**Important**: CLIP produces *single-vector* image embeddings (no late interaction),
so we'll simulate token-level embeddings by splitting the image into patches and
encoding each separately — a pedagogical approximation of what ColPali does natively.

In [ ]:
USE_COLPALI = False
colpali_model = None

try:
    from transformers import ColPaliForRetrieval, ColPaliProcessor
    print('ColPali classes available in transformers. Attempting to load vidore/colpali-v1.3...')
    colpali_processor = ColPaliProcessor.from_pretrained('vidore/colpali-v1.3', cache_dir=CACHE_DIR)
    colpali_model = ColPaliForRetrieval.from_pretrained(
        'vidore/colpali-v1.3',
        torch_dtype=torch.float16,
        cache_dir=CACHE_DIR
    ).to(device).eval()
    USE_COLPALI = True
    print('✓ ColPali loaded successfully!')
except Exception as e:
    print(f'ColPali not available: {e}')
    print('→ Falling back to CLIP for vision-language embedding demonstration.')
    USE_COLPALI = False

print(f'\nUsing: {"ColPali (vidore/colpali-v1.3)" if USE_COLPALI else "CLIP (openai/clip-vit-base-patch32)"}')

In [ ]:
if not USE_COLPALI:
    from transformers import CLIPModel, CLIPProcessor

    clip_model = CLIPModel.from_pretrained(
        'openai/clip-vit-base-patch32',
        cache_dir=CACHE_DIR
    ).to(device).eval()
    clip_processor = CLIPProcessor.from_pretrained(
        'openai/clip-vit-base-patch32',
        cache_dir=CACHE_DIR
    )
    print(f'✓ CLIP loaded  •  Vision dim: {clip_model.config.projection_dim}')
    print(f'  Patch size: 32×32  →  for 224×224 input: 7×7 = 49 patches')
else:
    print('Using ColPali — CLIP not needed.')

### Patch-Level Image Embeddings (Simulating ColPali)

ColPali produces one embedding vector per image patch natively. With CLIP, we
approximate this by:

1. Splitting each page image into a grid of overlapping crops
2. Encoding each crop independently with CLIP's vision encoder
3. Treating the resulting matrix as our "document token embeddings"

This isn't how ColPali actually works internally (it uses the ViT's patch
embeddings directly), but it captures the *spirit* of the approach:
**each spatial region of the page gets its own vector**.

In [ ]:
def get_page_patch_embeddings(page_img: Image.Image, grid_size: int = 4) -> np.ndarray:
    """
    Create patch-level embeddings for a page image.
    If ColPali: use native token-level output.
    If CLIP: split into grid_size×grid_size crops and encode each.
    Returns: (n_patches, embed_dim) numpy array
    """
    if USE_COLPALI:
        inputs = colpali_processor(images=page_img, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = colpali_model(**inputs)
        embs = outputs.embeddings[0].cpu().numpy()  # (n_patches, d)
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
        return (embs / norms).astype(np.float32)
    else:
        w, h = page_img.size
        crop_w, crop_h = w // grid_size, h // grid_size
        crops = []
        for row in range(grid_size):
            for col in range(grid_size):
                x0, y0 = col * crop_w, row * crop_h
                crop = page_img.crop((x0, y0, x0 + crop_w, y0 + crop_h))
                crops.append(crop)

        # Also encode the full page as an extra "global" patch
        crops.append(page_img)

        inputs = clip_processor(images=crops, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            embs = clip_model.get_image_features(**inputs)
        embs = embs.cpu().numpy()
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
        return (embs / norms).astype(np.float32)


def get_text_query_embedding(query: str) -> np.ndarray:
    """
    Encode query text into the vision-language shared space.
    Returns: (1, embed_dim) for single-vector or (n_tokens, embed_dim) for ColPali.
    """
    if USE_COLPALI:
        inputs = colpali_processor(text=query, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = colpali_model(**inputs)
        embs = outputs.embeddings[0].cpu().numpy()
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
        return (embs / norms).astype(np.float32)
    else:
        inputs = clip_processor(text=[query], return_tensors='pt', padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            embs = clip_model.get_text_features(**inputs)
        embs = embs.cpu().numpy()
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-10
        return (embs / norms).astype(np.float32)

# Test with first page
test_patch_embs = get_page_patch_embeddings(doc_pages[0])
test_query_emb = get_text_query_embedding('neural architecture search')
print(f'Page patch embeddings: {test_patch_embs.shape}')
print(f'Query embedding:       {test_query_emb.shape}')
print(f'\nEach page produces {test_patch_embs.shape[0]} patch vectors of dim {test_patch_embs.shape[1]}.')

In [ ]:
print('Encoding document pages into patch-level embeddings...')
page_patch_embs = []
for i, page in enumerate(doc_pages):
    t0 = time.time()
    embs = get_page_patch_embeddings(page)
    page_patch_embs.append(embs)
    print(f'  Page {i+1}: {embs.shape[0]:3d} patches × {embs.shape[1]}d  ({time.time()-t0:.2f}s)')

total_patches = sum(e.shape[0] for e in page_patch_embs)
print(f'\nTotal: {total_patches} patch vectors across {len(doc_pages)} pages')
print(f'Storage: {total_patches * page_patch_embs[0].shape[1] * 4 / 1024:.1f} KB (float32)')

### Visual Retrieval with MaxSim

Now we can do what ColPali does: given a text query, compare it against each page's
patch embeddings using MaxSim scoring. Even with CLIP as our backbone, this
demonstrates the concept: the query "finds" the most relevant spatial regions of each page.

In [ ]:
visual_queries = [
    'Table comparing NAS methods and their accuracy',
    'Bar chart showing search cost',
    'Weight sharing supernet training',
    'Future directions for architecture search',
    'DARTS accuracy on ImageNet',
]

print('=== Visual Retrieval (Image Patches + MaxSim) ===')
print()

for q in visual_queries:
    q_emb = get_text_query_embedding(q)  # (1, d) or (m, d)

    scores = []
    for idx, D in enumerate(page_patch_embs):
        if q_emb.shape[0] == 1:
            # CLIP: single query vector — compute dot product with each patch, take max
            s = float((q_emb @ D.T).max())
        else:
            # ColPali: full late interaction
            s = maxsim_score(q_emb, D)
        scores.append((s, idx))
    scores.sort(reverse=True)

    print(f'Q: "{q}"')
    for rank, (score, idx) in enumerate(scores[:3]):
        print(f'   #{rank+1}  (score={score:.4f}) Page {idx+1}: {page_descriptions[idx]}')
    print()

---
## 11 · Advantages of Visual Retrieval

ColPali's visual approach has several key advantages over text-based methods:

### 1. Tables Are First-Class Citizens
OCR on tables is notoriously error-prone: column alignment breaks, merged cells confuse parsers,
and numeric data gets garbled. ColPali sees the table *as an image* — structure preserved.

### 2. Figures and Charts Need No Description
A bar chart of "GPU-days vs accuracy" is immediately meaningful as patches. No need for a
captioning model to describe it (and inevitably miss details).

### 3. Layout Conveys Meaning
Headers, footnotes, sidebars, multi-column layouts — all carry semantic information that
plain text extraction destroys. ColPali preserves spatial relationships.

### 4. Language-Agnostic
ColPali works on Arabic, Chinese, mathematical notation — anything that can be rendered as
pixels. No language-specific tokenizer or OCR engine needed.

### 5. End-to-End Simplicity
```
Traditional: PDF → OCR → text cleanup → chunking → embedding → index → retrieve
ColPali:     PDF → render pages → embed pages → index → retrieve
```
Fewer stages = fewer failure modes.

---
## 12 · Building a FAISS Index for Patch Embeddings

In a production ColPali system, we'd store all patch embeddings in an index
for fast approximate nearest neighbor search. Let's build this and demonstrate
a two-stage retrieval pipeline:

1. **Stage 1 (ANN)**: Find candidate pages by matching query to nearest patches
2. **Stage 2 (MaxSim)**: Re-rank candidates with full late interaction scoring

In [ ]:
# Flatten all patch embeddings with page-id metadata
all_patches = np.vstack(page_patch_embs)
patch_to_page = []
for page_idx, embs in enumerate(page_patch_embs):
    patch_to_page.extend([page_idx] * embs.shape[0])
patch_to_page = np.array(patch_to_page)

print(f'Total patches: {all_patches.shape[0]}, dim: {all_patches.shape[1]}')

# Build FAISS index
patch_dim = all_patches.shape[1]
patch_index = faiss.IndexFlatIP(patch_dim)
patch_index.add(all_patches)
print(f'FAISS patch index: {patch_index.ntotal} vectors')

In [ ]:
def two_stage_retrieve(query: str, k_patches: int = 10, top_k_pages: int = 3):
    """
    Stage 1: Find top-k nearest patches via FAISS ANN.
    Stage 2: Re-rank the candidate pages with MaxSim.
    """
    q_emb = get_text_query_embedding(query)

    # Stage 1: ANN search for nearest patches
    if q_emb.shape[0] == 1:
        scores, indices = patch_index.search(q_emb.astype(np.float32), k_patches)
        candidate_pages = set(patch_to_page[indices[0]].tolist())
    else:
        # For multi-token queries, search with mean query vector
        q_mean = q_emb.mean(axis=0, keepdims=True)
        scores, indices = patch_index.search(q_mean.astype(np.float32), k_patches)
        candidate_pages = set(patch_to_page[indices[0]].tolist())

    # Stage 2: MaxSim re-ranking over candidate pages only
    results = []
    for page_idx in candidate_pages:
        D = page_patch_embs[page_idx]
        if q_emb.shape[0] == 1:
            s = float((q_emb @ D.T).max())
        else:
            s = maxsim_score(q_emb, D)
        results.append((s, page_idx))

    results.sort(reverse=True)
    return results[:top_k_pages]


print('=== Two-Stage Retrieval (ANN + MaxSim Re-rank) ===')
print()
for q in visual_queries:
    results = two_stage_retrieve(q)
    print(f'Q: "{q}"')
    for rank, (score, idx) in enumerate(results):
        print(f'   #{rank+1}  (score={score:.4f}) Page {idx+1}: {page_descriptions[idx]}')
    print()

---
## 13 · Loading the LLM: Qwen3-8B (4-bit)

For the generation phase of our RAG pipeline, we load `Qwen/Qwen3-8B` in 4-bit
NF4 quantization. This reduces VRAM usage from ~28 GB to ~8 GB while maintaining
strong reasoning ability.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

LLM_ID = 'Qwen/Qwen3-8B'
print(f'Loading {LLM_ID} in 4-bit NF4...')

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE_DIR)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    quantization_config=bnb_config,
    device_map='auto',
    cache_dir=CACHE_DIR,
)

print(f'✓ {LLM_ID} loaded')
print(f'  Parameters: ~14B (4-bit quantized)')
print(f'  Memory footprint: ~{llm_model.get_memory_footprint() / 1e9:.1f} GB')

In [ ]:
def generate(prompt: str, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    """Generate text with Qwen3-8B."""
    messages = [{'role': 'user', 'content': prompt}]
    text = llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = llm_tokenizer(text, return_tensors='pt').to(llm_model.device)
    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
        )
    new_tokens = output_ids[0][inputs['input_ids'].shape[1]:]
    return llm_tokenizer.decode(new_tokens, skip_special_tokens=True)

# Quick test
test_response = generate('In one sentence, what is neural architecture search?', max_new_tokens=100)
print(f'LLM test: {test_response}')

---
## 14 · Complete Multimodal RAG Pipeline

Now we wire everything together into the full pipeline:

```
User Query
    │
    ├──→ Encode query (text encoder)
    │
    ├──→ Retrieve pages (FAISS ANN → MaxSim re-rank)
    │
    ├──→ Get page descriptions as context
    │     (in real ColPali: feed page images to VLM)
    │
    └──→ Generate answer (Qwen3 LLM)
```

Since we have synthetic pages (not real PDFs), we use the page descriptions
as our context. In a real system, the retrieved page *images* would be fed
to a vision-language model for generation.

In [ ]:
# Extended text content for each page (simulating what OCR/VLM would extract)
page_contexts = [
    'This page is the abstract and introduction of a survey on Neural Architecture Search (NAS). '
    'It covers the three main components: search space, search strategy, and performance estimation. '
    'Search spaces include chain-structured, multi-branch, and cell-based designs. '
    'Search strategies include RL (Zoph & Le 2017), evolutionary (Real et al 2019), '
    'and gradient-based (DARTS, Liu et al 2019). Performance estimation bottleneck '
    'is addressed by weight sharing, early stopping, and predictor-based methods.',

    'Table 1 comparing NAS methods: '
    'NASNet: 1800 GPU-days, 97.35% CIFAR-10, 74.0% ImageNet. '
    'ENAS: 0.5 GPU-days, 97.11% CIFAR-10. '
    'DARTS: 1.5 GPU-days, 97.24% CIFAR-10, 73.3% ImageNet. '
    'ProxylessNAS: 8.3 GPU-days, 75.1% ImageNet. '
    'EfficientNet: 84.3% ImageNet. '
    'OFA: 2.0 GPU-days, 80.0% ImageNet. '
    'FBNet: 9.0 GPU-days, 74.9% ImageNet. '
    'MnasNet: 288 GPU-days, 75.2% ImageNet.',

    'Figure 1: Bar chart comparing NAS methods by ImageNet top-1 accuracy. '
    'EfficientNet leads at 84.3%, followed by OFA at 80.0%. '
    'ProxylessNAS and MnasNet are around 75%. DARTS and NASNet are around 73-74%.',

    'Section on weight sharing and one-shot NAS methods. '
    'One-shot methods train a supernet containing all candidate architectures. '
    'Different subgraphs are sampled without retraining from scratch. '
    'Reduces cost from thousands of GPU-days to single digits. '
    'Few-shot NAS (Zhao et al 2021) splits supernet to reduce interference. '
    'Hardware-aware NAS incorporates latency, memory, and energy constraints.',

    'Conclusions and future directions: '
    '1. Transferable architectures across tasks and datasets. '
    '2. Beyond classification: detection, segmentation, NLP, multimodal. '
    '3. Green AI: reducing carbon footprint. '
    '4. Foundation model adaptation: NAS for LLM fine-tuning. '
    '5. Theory: connecting NAS to scaling laws and loss landscapes.',
]

def rag_query(question: str, top_k: int = 2) -> str:
    """Full RAG pipeline: retrieve → construct prompt → generate."""
    # Retrieve
    results = two_stage_retrieve(question, k_patches=10, top_k_pages=top_k)
    retrieved_pages = [(idx, page_contexts[idx]) for _, idx in results]

    # Construct prompt
    context_str = '\n\n'.join(
        f'[Page {idx+1}]: {ctx}' for idx, ctx in retrieved_pages
    )
    prompt = (
        f'You are a helpful research assistant. Answer the question based on the '
        f'retrieved document pages below.\n\n'
        f'=== Retrieved Context ===\n{context_str}\n\n'
        f'=== Question ===\n{question}\n\n'
        f'Provide a clear, detailed answer citing specific pages when possible.'
    )

    # Generate
    answer = generate(prompt, max_new_tokens=300)
    return answer, retrieved_pages

print('✓ RAG pipeline ready.')

In [ ]:
rag_questions = [
    'What is the ImageNet top-1 accuracy of DARTS and how many GPU-days does it take?',
    'What are the future directions mentioned in the survey?',
    'How does weight sharing reduce NAS search cost?',
]

for q in rag_questions:
    print(f'╔══ Question: {q}')
    answer, pages = rag_query(q)
    print(f'║  Retrieved: {[", ".join(f"Page {idx+1}" for idx, _ in pages)]}')
    print(f'║')
    for line in textwrap.wrap(answer, width=80):
        print(f'║  {line}')
    print(f'╚{"═"*78}')
    print()

---
## 15 · Comparison: Captioning-Based vs Embedding-Based Multimodal RAG

| Dimension | Captioning-Based | Embedding-Based (ColPali) |
|---|---|---|
| **Pipeline** | PDF → OCR → Caption → Chunk → Embed → Retrieve | PDF → Render → Embed patches → Retrieve |
| **Information loss** | High (OCR errors, caption bias, chunking artifacts) | Low (raw pixels preserved) |
| **Table handling** | Fragile (OCR struggles with alignment) | Natural (table is a grid of patches) |
| **Figure handling** | Requires separate captioning | Natively encoded |
| **Layout preservation** | Lost | Preserved |
| **Granularity** | Chunk-level (typically 512 tokens) | Patch-level (16×16 pixels) |
| **Matching** | Single-vector cosine | Late interaction MaxSim |
| **Storage** | 1 vector per chunk | ~1000 vectors per page |
| **Compute (indexing)** | Cheap (text encoder) | Expensive (vision encoder) |
| **Compute (query)** | Cheap (1 dot product per chunk) | Medium (MaxSim over patches) |
| **Maturity** | Established, many tools | Cutting edge (2024) |

### Computational Cost Analysis

Let's quantify the storage and compute tradeoffs between the two approaches.

In [ ]:
# ── Cost comparison (realistic numbers) ──

n_pages = 1000
chunks_per_page = 3  # average text chunks per page
patches_per_page = 1024  # ColPali: ~32×32 patch grid
text_dim = 768   # BGE
patch_dim = 128  # ColPali (compressed)
bytes_per_float = 4

# Storage
text_storage_mb = n_pages * chunks_per_page * text_dim * bytes_per_float / 1e6
patch_storage_mb = n_pages * patches_per_page * patch_dim * bytes_per_float / 1e6

# MaxSim operations per query
query_tokens = 10
text_ops = n_pages * chunks_per_page * 1  # cosine sim per chunk
patch_ops = n_pages * patches_per_page * query_tokens  # full MaxSim

print(f'=== Cost Comparison for {n_pages} Pages ===')
print()
print(f'{"Metric":<30} {"Text-Based":>15} {"ColPali":>15}')
print('─' * 62)
print(f'{"Vectors stored":<30} {n_pages * chunks_per_page:>15,} {n_pages * patches_per_page:>15,}')
print(f'{"Storage (MB)":<30} {text_storage_mb:>15.1f} {patch_storage_mb:>15.1f}')
print(f'{"Dot products per query":<30} {text_ops:>15,} {patch_ops:>15,}')
print(f'{"Storage ratio":<30} {"1x":>15} {patch_storage_mb/text_storage_mb:>14.0f}x')
print(f'{"Compute ratio":<30} {"1x":>15} {patch_ops/text_ops:>14.0f}x')
print()
print('→ ColPali trades ~56x more storage and ~3400x more compute for')
print('  dramatically better visual document understanding.')
print('  In practice, ANN indexing + re-ranking makes this tractable.')

---
## 16 · Synthesis: ColPali and the Future of Document Retrieval

### What We've Learned

1. **Late interaction (MaxSim)** is more expressive than single-vector cosine similarity.
   Each query token independently finds its best match in the document, enabling
   fine-grained retrieval that single vectors can't achieve.

2. **ColBERT** pioneered late interaction for text. It keeps token-level embeddings
   for both query and document, scoring via MaxSim.

3. **ColPali** extends ColBERT to vision: document pages are embedded as images
   (patch-level vectors), and text queries interact with these patches via the
   same MaxSim mechanism.

4. **Visual retrieval** preserves tables, figures, layout, and multilingual content
   that OCR/captioning pipelines lose.

### When ColPali Wins

- **Document-heavy domains**: Legal, medical, financial — lots of tables and structured layouts.
- **Multilingual/mixed-script**: No need for language-specific OCR.
- **Figure-rich content**: Scientific papers, patents, technical manuals.
- **Quality-sensitive applications**: Where OCR errors are unacceptable.

### When Text-Based is Better

- **Pure text documents**: Blog posts, articles — no visual structure to exploit.
- **Resource-constrained**: ColPali needs ~56x more storage and significant GPU for encoding.
- **Existing infrastructure**: If you already have a mature text-based RAG pipeline.

### Future Directions

- **Efficiency**: ColBERTv2-style compression for patch embeddings (ongoing).
- **Multi-page reasoning**: Attending across multiple pages, not just retrieving individual ones.
- **Hybrid approaches**: Using ColPali for retrieval but text-based models for generation.
- **Training data**: More diverse document training data (receipts, handwriting, blueprints).
- **Integration**: ColPali as a drop-in retriever for existing RAG frameworks.

ColPali represents a fundamental shift in how we think about document retrieval:
**documents are visual objects**, and treating them as such unlocks capabilities
that text-only approaches cannot match.

In [ ]:
# Free GPU memory
import gc

for name in ['llm_model', 'bge_model', 'clip_model', 'colpali_model']:
    if name in dir():
        obj = eval(name)
        if obj is not None:
            del obj

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')
else:
    print('Cleanup complete (CPU mode).')

print('\n✓ Notebook complete.')